In [ ]:
from pandapower.plotting.plotly import simple_plotly
import pandapower.plotting.plotly as pplotly
from pandas import Series
import pandapower as pp
import pandapower.networks as pn
net = pn.case_ieee30()
pp.rundcpp(net)
simple_plotly(net);

In [ ]:
lc = pplotly.create_line_trace(net,net.line.index, color='black',
                               infofunc=Series(index=net.line.index,
                                              data='Line ' + net.line.index.astype(str) + '<br>'\
                                                   +'fb: ' + net.line.from_bus.astype(str)+' tb: ' + net.line.to_bus.astype(str))
                                              )
bc = pplotly.create_bus_trace(net, net.bus.index, size=10, color="orange",
                              infofunc=Series(index=net.bus.index,
                                              data='Bus' + net.bus.name.astype(str) + '<br>' + net.bus.vn_kv.astype(str) + ' kV'))
tc = pplotly.create_trafo_trace(net,net.trafo.index, color='green',
                                infofunc=Series(index=net.trafo.index,
                                              data='Trafo ' + net.trafo.index.astype(str) + '<br>'\
                                                   +'hv: ' + net.trafo.hv_bus.astype(str)+' lv: ' + net.trafo.lv_bus.astype(str))
                                              )
                                
pplotly.draw_traces(tc+ lc + bc, figsize=1, aspectratio=(8,6));

In [ ]:
import pandapower as pp
import pandapower.networks as nw
import pandapower.plotting as plot
%matplotlib inline

net = nw.mv_oberrhein()
pp.runpp(net)
cmap_list=[(20, "green"), (50, "yellow"), (60, "red")]
cmap, norm = plot.cmap_continuous(cmap_list)
lc = plot.create_line_collection(net, net.line.index, zorder=1, cmap=cmap, norm=norm, linewidths=2)
plot.draw_collections([lc], figsize=(8,6))

In [ ]:
net = nw.mv_oberrhein()
pp.runpp(net)
cmap_list=[((0.975, 0.985), "blue"), ((0.985, 1.0), "green"), ((1.0, 1.03), "red")]
cmap, norm = plot.cmap_discrete(cmap_list)
bc = plot.create_bus_collection(net, net.bus.index, size=80, zorder=2, cmap=cmap, norm=norm)

cmap_list=[((10, 40), "green"), ((40, 55), "yellow"), ((55, 60), "red")]
cmap, norm = plot.cmap_discrete(cmap_list)
lc = plot.create_line_collection(net, net.line.index, zorder=1, cmap=cmap, norm=norm, linewidths=2)
plot.draw_collections([lc, bc], figsize=(8,6))

In [1]:
import numpy as np
import plotly.graph_objects as go

# 1. prepare “phases” for animation
phases = np.linspace(0, 2*np.pi, 30)

# 2. build the initial figure
fig = go.Figure(
    data=[go.Scatter(
        x=np.linspace(0, 2*np.pi, 100),
        y=np.sin(np.linspace(0, 2*np.pi, 100) + phases[0]),
        mode='lines'
    )],
    layout=go.Layout(
        title="Simple Sine‐Wave Animation",
        xaxis=dict(range=[0, 2*np.pi], autorange=False),
        yaxis=dict(range=[-1, 1], autorange=False),
        updatemenus=[{
            "type": "buttons",
            "buttons": [{
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {"frame": {"duration": 100, "redraw": True},
                     "fromcurrent": True}
                ]
            }]
        }]
    ),
    # 3. define one frame per phase shift
    frames=[
        go.Frame(
            data=[go.Scatter(
                x=np.linspace(0, 2*np.pi, 100),
                y=np.sin(np.linspace(0, 2*np.pi, 100) + φ),
                mode='lines'
            )],
            name=str(i)
        )
        for i, φ in enumerate(phases)
    ]
)

# 4. add a slider to scrub through frames
sliders = [{
    "currentvalue": {"prefix": "Phase: "},
    "steps": [
        {
            "args": [[fr.name], {"frame": {"duration": 0, "redraw": True}}],
            "label": fr.name,
            "method": "animate"
        } for fr in fig.frames
    ]
}]
fig.update_layout(sliders=sliders)

# show it
fig.show()


In [ ]:
import plotly.io as pio
import imageio

# 1) render each frame to a PNG in memory
imgs = []
for frame in fig.frames:
    fig.update(data=frame.data)               # load the frame’s data
    img_bytes = pio.to_image(fig, format="png",
                             width=800, height=500)
    imgs.append(imageio.imread(img_bytes))

# 2a) write a GIF
imageio.mimsave("sine_wave.gif", imgs, fps=10)

# 2b) (optional) write an MP4 using ffmpeg via imageio
# imageio.mimsave("sine_wave.mp4", imgs, fps=10,
#                 codec="libx264", quality=8)
